# 19.10 提升建模 / Uplift Modeling (S/T/X-learners)

**中文**：19.9 用因果森林估个体处理效应(CATE)。本节讲它在营销里的专门形态——**提升建模(Uplift Modeling)**,以及估 CATE 的**通用框架:元学习器(meta-learners, S/T/X/R-learner)**——它们能把**任意**机器学习模型(GBDT、神经网络)改造成 CATE 估计器。核心问题是营销的终极难题:*"我该给谁发优惠券?"* 答案不是"谁最可能买"(那是预测),而是"**谁因为发了券才会买**"(那是因果/uplift)。这个区别价值连城。
**English**: 19.9 used a causal forest to estimate individual effects (CATE). This section covers its specialized form in marketing — **Uplift Modeling** — and the **general framework for estimating CATE: meta-learners (S/T/X/R-learners)** — which turn **any** ML model (GBDT, neural nets) into a CATE estimator. The core is marketing's ultimate question: *"whom should I send a coupon?"* The answer is not "who is most likely to buy" (that's prediction) but "**who buys only because of the coupon**" (that's causal / uplift). This distinction is worth a fortune.

---

**中文**：营销把用户按"处理(发券)会怎么改变行为"分成**四类**,这是理解 uplift 的钥匙:
**English**: Marketing splits users into **four types** by "how treatment (a coupon) changes their behavior" — the key to understanding uplift:

| | 不发券也买 / buys w/o coupon | 不发券不买 / doesn't buy w/o |
|---|---|---|
| **发券会买** | 🟢 **稳买客 Sure Things**(发券纯浪费钱)| 🎯 **可劝说 Persuadables**(**只该发给他们!**)|
| **发券不买** | 🔴 **睡狗 Sleeping Dogs**(发券**反而惹烦**→别发!)| ⚪ **无望客 Lost Causes**(发了也白发)|

**中文**：**只有"可劝说(Persuadables)"值得发券**(uplift 为正);对"稳买客"和"无望客"发券是浪费;对"**睡狗**"发券**适得其反**(打扰导致退订/反感,uplift 为负)。传统"响应模型"(预测谁会买)会把券发给"稳买客"(他们响应率最高但本来就买)——**大错特错**。uplift 直接估 $\tau(x)=P(\text{买}|发券)-P(\text{买}|不发)$,只找券真正能改变行为的人。
**English**: **Only "Persuadables" are worth a coupon** (positive uplift); coupons to "Sure Things" and "Lost Causes" are wasted; coupons to "**Sleeping Dogs**" **backfire** (annoyance → unsubscribe/aversion, negative uplift). A traditional "response model" (predict who buys) would send coupons to "Sure Things" (highest response rate but they'd buy anyway) — **badly wrong**. Uplift directly estimates $\tau(x)=P(\text{buy}|coupon)-P(\text{buy}|no)$, targeting only those the coupon truly moves.

**中文**：**元学习器(meta-learners)** 是估 CATE 的通用配方,能套用任意基模型:
**English**: **Meta-learners** are general recipes for estimating CATE, usable with any base model:
- **S-learner(Single)**:训**一个**模型,把处理 $T$ 当一个特征;$\hat\tau(x)=f(x,1)-f(x,0)$。简单,但若模型忽视 $T$ 会低估效应。
  **S-learner (Single)**: train **one** model with treatment $T$ as a feature; $\hat\tau(x)=f(x,1)-f(x,0)$. Simple, but if the model underweights $T$ it underestimates the effect.
- **T-learner(Two)**:对处理组、对照组各训**一个**模型;$\hat\tau(x)=f_1(x)-f_0(x)$。直接,但两组样本不均时较差。
  **T-learner (Two)**: train **two** models, one per group; $\hat\tau(x)=f_1(x)-f_0(x)$. Direct, but poor when the groups are imbalanced.
- **X-learner**:T-learner 的改进——先算每个人的"插补处理效应",再对它建模,并用倾向得分加权。**处理组/对照组极不均衡时表现最好**(营销常见:只有少数人被处理)。
  **X-learner**: an improvement — first impute each unit's treatment effect, then model it, weighting by propensity. **Best when groups are very imbalanced** (common in marketing: few are treated).

> 💡 **面试速查 / Interview cheat-sheet（★★★ 营销DS必考）**
> **中文**：**Uplift=估 τ(x)=P(Y|处理)−P(Y|不处理)**, 找"因处理才改变行为"的人。**四类人**:可劝说(发)、稳买客(别发,浪费)、无望客(别发)、**睡狗(别发,负效应)**。**关键**:uplift ≠ 响应模型(预测谁会买会把券发给稳买客)。**元学习器**:**S**(单模型+T当特征)、**T**(两模型相减)、**X**(插补效应+倾向加权, 抗不均衡)、**R**(残差化+加权)。**评估用 Qini/AUUC 曲线**(不能用 AUC/RMSE, 因无个体真值)——按预测 uplift 排序, 看累计增量收益 vs 随机。**理想数据=随机实验**; 上线前用真实 A/B 验证定向策略。
> **English**: **Uplift = estimate τ(x)=P(Y|treat)−P(Y|no treat)**, finding those whose behavior changes because of treatment. **Four types**: Persuadables (target), Sure Things (don't, wasteful), Lost Causes (don't), **Sleeping Dogs (don't, negative effect)**. **Key**: uplift ≠ a response model (predicting who buys would coupon the Sure Things). **Meta-learners**: **S** (single model + T as feature), **T** (two models differenced), **X** (impute effects + propensity weighting, robust to imbalance), **R** (residualization + weighting). **Evaluate with Qini/AUUC curves** (not AUC/RMSE, since no individual ground truth) — rank by predicted uplift and plot cumulative incremental gain vs random. **Ideal data = randomized experiment**; validate the targeting policy with a real A/B before launch.


In [ ]:

# ============================================================
# 模拟营销数据:含四类人(尤其睡狗)/ marketing data with the four types (esp. sleeping dogs)
# 中文:随机发券(T)。真实 uplift 随特征变:x0高=可劝说(正), x1高=睡狗(负)。结果 Y=是否购买。
# English: random coupon (T). True uplift varies: high x0 = persuadable (positive), high x1 = sleeping dog (negative).
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor as GBR
rng=np.random.default_rng(0)
N,d=6000,6
X=rng.uniform(0,1,(N,d))
T=rng.integers(0,2,N)                                        # 随机发券 / random coupon
true_uplift = 1.5*(X[:,0]-0.5) - 1.2*(X[:,1]-0.5)          # 真实 uplift(有正有负=含睡狗)/ true uplift
base = 0.30 + 0.20*X[:,2]                                   # 基线购买率 / baseline buy prob
p = np.clip(base + true_uplift*T, 0.01, 0.99)
Y = (rng.random(N)<p).astype(float)                        # 是否购买 / purchase
tr=np.arange(4500); te=np.arange(4500,N)                    # 训练/测试 / train/test
print(f"随机发券给 {T.mean():.0%} 用户 / coupon to {T.mean():.0%}")
print(f"真实 uplift 范围: {true_uplift.min():.2f} 到 {true_uplift.max():.2f}  (负值=睡狗, 发券反而少买)")
print(f"整体平均 uplift / average uplift: {true_uplift.mean():+.3f}")


**中文**：注意:整体平均 uplift 接近 0(可劝说的正效应和睡狗的负效应相互抵消)!如果只看平均,你会以为"发券没用"而放弃。但**真相是有一批人该发、有一批人绝不能发**。下面用三种元学习器(S/T/X)把每个人的 uplift 估出来。
**English**: Note: the overall average uplift is near 0 (persuadables' positive and sleeping dogs' negative cancel)! Judging by the average alone, you'd conclude "coupons don't work" and give up. But **the truth is some should get them and some absolutely shouldn't**. Below we estimate each person's uplift with three meta-learners (S/T/X).


In [ ]:

# ============================================================
# 从零实现 S/T/X 三种元学习器 / S-, T-, X-learners from scratch
# ============================================================
# ① S-learner:单模型, 把 T 当特征 / single model with T as a feature
s_model = GBR(max_depth=3,n_estimators=150).fit(np.c_[X[tr],T[tr]], Y[tr])
cate_S = s_model.predict(np.c_[X[te],np.ones(len(te))]) - s_model.predict(np.c_[X[te],np.zeros(len(te))])

# ② T-learner:两个模型相减 / two models, differenced
m1 = GBR(max_depth=3).fit(X[tr][T[tr]==1], Y[tr][T[tr]==1])   # 处理组模型 / treated model
m0 = GBR(max_depth=3).fit(X[tr][T[tr]==0], Y[tr][T[tr]==0])   # 对照组模型 / control model
cate_T = m1.predict(X[te]) - m0.predict(X[te])

# ③ X-learner:插补效应 + 倾向加权 / impute effects + propensity weighting
tr1=tr[T[tr]==1]; tr0=tr[T[tr]==0]
D1 = Y[tr1] - m0.predict(X[tr1])                              # 处理组:实际 - 反事实估计 / imputed effect (treated)
D0 = m1.predict(X[tr0]) - Y[tr0]                              # 对照组:反事实估计 - 实际 / imputed effect (control)
tau1 = GBR(max_depth=3).fit(X[tr1], D1)                       # 对插补效应建模 / model the imputed effects
tau0 = GBR(max_depth=3).fit(X[tr0], D0)
g=0.5                                                         # 倾向得分(随机→0.5)/ propensity
cate_X = g*tau0.predict(X[te]) + (1-g)*tau1.predict(X[te])    # 加权组合 / weighted combination

tau_te = true_uplift[te]
print(f"{'学习器/learner':<16}{'与真实uplift相关性':>18}")
for name,c in [("S-learner",cate_S),("T-learner",cate_T),("X-learner",cate_X)]:
    print(f"{name:<16}{np.corrcoef(tau_te,c)[0,1]:>18.3f}")


**中文**：三种学习器都不错地估出了 uplift。但**怎么评估一个 uplift 模型好不好？** 不能用 AUC/RMSE——因为**没有个体真值**(每人只观测到一种结果)。要用 **Qini 曲线**:按预测 uplift 从高到低排序,依次"处理",看累计**增量购买**(比随机排序多带来的净购买)。曲线越高、越靠上,模型越能把券精准发给可劝说的人。
**English**: All three learners estimate uplift well. But **how do you evaluate an uplift model?** Not with AUC/RMSE — there is **no individual ground truth** (each person shows only one outcome). Use the **Qini curve**: rank by predicted uplift high-to-low, "treat" in that order, and track cumulative **incremental purchases** (net purchases beyond random ordering). The higher the curve, the better the model targets coupons at persuadables.


In [ ]:

# ============================================================
# Qini 曲线评估 + 四类人可视化 / Qini evaluation + four-types visualization
# ============================================================
def qini_curve(cate_hat):
    order=np.argsort(cate_hat)[::-1]                          # 按预测 uplift 降序 / sort by predicted uplift
    Tt=T[te][order]; Yt=Y[te][order]
    nt=np.cumsum(Tt); nc=np.cumsum(1-Tt)
    st=np.cumsum(Tt*Yt); sc=np.cumsum((1-Tt)*Yt)
    return st - sc*nt/np.maximum(nc,1)                        # Qini = 处理组购买 - 缩放的对照组购买
fracs=np.arange(1,len(te)+1)/len(te)

fig,ax=plt.subplots(1,3,figsize=(17,4.7))
# ① 四类人象限(真实 uplift 上色)/ four-types quadrant
sc=ax[0].scatter(base[te],true_uplift[te],c=true_uplift[te],cmap="RdYlGn",s=8,alpha=0.5,vmin=-1,vmax=1)
ax[0].axhline(0,color="k",lw=0.8); ax[0].axvline(0.5,color="gray",ls=":")
ax[0].text(0.7,0.6,"🎯可劝说\nPersuadable",fontsize=9,ha="center"); ax[0].text(0.7,-0.7,"🔴睡狗\nSleeping Dog",fontsize=9,ha="center")
ax[0].set_title("四类人:只该发给可劝说(绿), 别发睡狗(红)"); ax[0].set_xlabel("基线购买率"); ax[0].set_ylabel("真实 uplift"); plt.colorbar(sc,ax=ax[0],fraction=0.046)
# ② Qini 曲线 / Qini curves
for name,c,col in [("S-learner",cate_S,"#4C72B0"),("T-learner",cate_T,"#55A868"),("X-learner",cate_X,"#8172B3")]:
    q=qini_curve(c); ax[1].plot(fracs,q,color=col,label=f"{name}")
q_rand=qini_curve(rng.random(len(te))); ax[1].plot(fracs,q_rand,"k--",label="随机 random")
ax[1].set_title("Qini 曲线:越高越好(精准找可劝说)/ Qini curves"); ax[1].set_xlabel("处理人群比例"); ax[1].set_ylabel("累计增量购买"); ax[1].legend(fontsize=8)
# ③ uplift 模型 vs 响应模型(谁该发券)/ uplift vs response model
resp = m1.predict(X[te])                                      # 响应模型:预测购买概率(=稳买客最高)/ response model
ax[2].scatter(resp,cate_T,s=8,alpha=0.3,color="#4C72B0")
ax[2].axhline(0,color="r",ls="--");
ax[2].set_title("响应≠uplift:高响应(稳买客)uplift可能≈0 / response ≠ uplift"); ax[2].set_xlabel("响应模型:P(购买)"); ax[2].set_ylabel("uplift 估计")
plt.tight_layout(); plt.savefig("/tmp/ci10_viz.png",dpi=80); plt.show()
half=len(te)//2
print(f"Qini@处理一半人群: S={qini_curve(cate_S)[half]:.0f}, T={qini_curve(cate_T)[half]:.0f}, X={qini_curve(cate_X)[half]:.0f} vs 随机={q_rand[half]:.0f}")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **uplift ≠ 响应,这是营销数据科学最值钱的认知**:传统"响应模型"预测"谁会买",会把优惠券发给**稳买客**(他们购买概率最高)——但这些人**本来就会买**,发券纯属烧钱。uplift 直接估"发券带来的**增量**",只找**可劝说**的人。右图清楚显示:响应概率高的人,uplift 可能≈0(稳买客)。**把预算从'高响应'挪到'高 uplift',同样的钱能多带来一大截增量销售。**
2. **睡狗是最容易被忽视、代价最惨的一类**:有些人被打扰后**反而不买了**(uplift 为负)。传统方法根本发现不了他们(甚至因为他们基线购买率不低而误发)。uplift 能识别负效应人群,**主动避开**——这在推送、召回、涨价通知里极其重要(发错了会导致退订、流失)。整体平均 uplift 接近 0 正是因为可劝说和睡狗相互抵消——**只看平均会让你错误地放弃一个其实很有价值的策略**。
3. **评估必须用 Qini/AUUC,不能用准确率**:因果的根本困难是没有个体真值(每人只观测一种结果),所以 AUC/RMSE 用不了。Qini 曲线通过"按预测排序、比较处理组与对照组的累计差异"来间接评估——曲线越高,模型越会"把券发给对的人"。**诚实局限**:①元学习器的质量高度依赖基模型和数据量(uplift 信噪比低,需大样本);②S-learner 在效应弱时容易把 uplift"正则化没了"(低估);③**理想数据是随机实验**,观测数据要先去混杂;④**最终一定要用一个真实 A/B 验证你的定向策略**(离线 Qini 好≠上线真赚钱)。

**English**:
1. **uplift ≠ response — the most valuable insight in marketing data science**: a traditional "response model" predicts "who buys" and would coupon the **Sure Things** (highest buy probability) — but they'd **buy anyway**, so it's pure burn. Uplift estimates the **incremental** effect of the coupon, targeting only **persuadables**. The right plot shows clearly: high-response people can have uplift ≈0 (Sure Things). **Shifting budget from "high response" to "high uplift" brings far more incremental sales for the same money.**
2. **Sleeping Dogs are the most overlooked and costly type**: some people **stop buying when annoyed** (negative uplift). Traditional methods can't find them (and may even coupon them since their baseline isn't low). Uplift identifies negative-effect groups and **actively avoids them** — crucial for push notifications, win-back, and price-change alerts (a wrong send causes unsubscribes/churn). The overall average uplift near 0 is exactly because persuadables and sleeping dogs cancel — **judging by the average would wrongly make you abandon a genuinely valuable policy.**
3. **Evaluation must use Qini/AUUC, not accuracy**: the fundamental difficulty is no individual ground truth (each person shows one outcome), so AUC/RMSE don't apply. The Qini curve evaluates indirectly via "rank by prediction, compare cumulative treated-vs-control differences" — higher curve = better targeting. **Honest limits**: ① meta-learner quality depends heavily on the base model and data size (uplift has low signal-to-noise, needs large samples); ② the S-learner tends to "regularize the uplift away" when the effect is weak (underestimating); ③ **the ideal data is a randomized experiment**; observational data must be de-confounded; ④ **always validate the targeting policy with a real A/B** (good offline Qini ≠ real online profit).

> 💼 **实战视角 / Practical angle**
> **中文**:uplift 是**精准营销/增长的核心引擎**:优惠券/补贴定向、push 推送(避开睡狗)、留存挽回、涨价通知、广告定向。落地流程:①**用随机实验采数据**(随机发/不发);②训 uplift 模型(**X-learner 或因果森林**常最稳, 尤其处理组少时);③**Qini/AUUC 选模型**;④按预测 uplift 排序, 定"发给 top-k%"的阈值(结合 ROI);⑤**上线前用一个真实 A/B 验证**定向策略真能提升增量 ROI。工具:`causalml`(Uber)、`econml`(微软)、`scikit-uplift`。面试金句:*"uplift 估的是发券带来的增量 τ(x)=P(Y|发)−P(Y|不发), 只发给'可劝说'、避开'睡狗', 和'响应模型'(会浪费在稳买客上)本质不同; 用 S/T/X 元学习器估, 用 Qini/AUUC 评估(没个体真值), 最后用真实 A/B 验证。"*
> **English**: Uplift is the **core engine of precision marketing/growth**: coupon/subsidy targeting, push notifications (avoid Sleeping Dogs), retention win-back, price-change alerts, ad targeting. Workflow: ① **collect data via a randomized experiment** (random treat/not); ② train an uplift model (**X-learner or causal forest** are often most robust, especially with few treated); ③ **pick the model by Qini/AUUC**; ④ rank by predicted uplift and set a "target the top-k%" threshold (with ROI); ⑤ **validate the targeting policy with a real A/B before launch**. Tools: `causalml` (Uber), `econml` (Microsoft), `scikit-uplift`. Interview line: *"Uplift estimates a coupon's incremental effect τ(x)=P(Y|treat)−P(Y|no), targeting Persuadables and avoiding Sleeping Dogs — fundamentally different from a response model (which wastes on Sure Things); estimate with S/T/X meta-learners, evaluate with Qini/AUUC (no individual ground truth), and validate with a real A/B."*

---
### 小结 / Summary
- **中文**:uplift=估处理的增量效应 τ(x); 四类人只发可劝说、避开睡狗; uplift≠响应模型。
- **English**: Uplift = estimate treatment's incremental effect τ(x); of the four types, target Persuadables and avoid Sleeping Dogs; uplift ≠ response model.
- **中文**:元学习器 S(单模型)/T(两模型)/X(插补+倾向加权, 抗不均衡) 把任意模型变成 CATE 估计器。
- **English**: Meta-learners S (single) / T (two) / X (impute + propensity weighting, robust to imbalance) turn any model into a CATE estimator.
- **中文**:评估用 Qini/AUUC(无个体真值); 需随机数据; 上线前用真实 A/B 验证定向策略。
- **English**: Evaluate with Qini/AUUC (no individual ground truth); needs randomized data; validate the targeting policy with a real A/B.
